# 🩺 Healthcare QLoRA Fine-Tuning — Google Colab (Free GPU)

Fine-tune **Phi-3-mini** (or **Llama-3.2-3B**) on medical Q&A with **QLoRA**, then download the trained LoRA adapter and serve it locally with the project's Docker stack.

### Before you start
1. **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (free tier works).
2. Run the cells top to bottom.
3. At the end you'll download `healthcare-qlora-adapter.zip`. Unzip it into your local project's `outputs/` folder, set `BASE_MODEL` back to Phi-3/Llama in `.env`, and `docker compose up -d` — the API will serve your fine-tuned model.

> ⚠️ This adapter is trained on a 3.8B base model and needs the **same base model** at serve time. Serving it on CPU works but is slow; a GPU box is recommended for the full-size model.

*Estimated time on a T4: ~20–40 min for a 2–3k example subset, 1 epoch.*

## 1. Verify the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU! Go to Runtime > Change runtime type > T4 GPU, then re-run.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install dependencies

Pinned to the same versions as the project (`requirements.txt`) so training matches the repo exactly.

In [ ]:
%pip install -q \
    transformers==4.43.3 \
    peft==0.12.0 \
    trl==0.9.6 \
    accelerate==0.33.0 \
    bitsandbytes==0.43.1 \
    datasets==2.20.0 \
    sentencepiece==0.2.0
print('\n✅ Installed. If Colab asks to restart the runtime, do it, then re-run from cell 1.')

## 3. Configuration

These mirror `src/config.py`. Tweak `MAX_SAMPLES` / `NUM_EPOCHS` to trade quality for speed.

In [ ]:
# ---- Model ----
# Open, no login needed:
BASE_MODEL = 'microsoft/Phi-3-mini-4k-instruct'
# Gated (uncomment + run the HF login cell + accept the license on HF):
# BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'

OUTPUT_DIR = 'outputs/healthcare-qlora-adapter'

# ---- Data ----
MAX_SAMPLES = 3000   # raw rows to use (None = full dataset, much slower)
SEED = 42

# ---- LoRA (QLoRA) ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
    'qkv_proj', 'gate_up_proj',  # Phi-3 fused names
]

# ---- Training ----
NUM_EPOCHS = 1
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8           # effective batch size = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024

INSTRUCTION = (
    "You are a knowledgeable medical assistant. Answer the patient's health "
    "question accurately, clearly, and safely. If the question requires "
    "professional diagnosis, advise consulting a healthcare provider."
)
print('Config set. Base model:', BASE_MODEL)

## 4. (Optional) Hugging Face login

Only needed for **gated** models like Llama-3.2. Skip for Phi-3.

In [ ]:
# from huggingface_hub import login
# login()  # paste a token from https://huggingface.co/settings/tokens

## 5. Load + preprocess the dataset

MedQuAD medical Q&A (with a PubMedQA fallback). Clean → dedupe → instruction format → split. This is a compact version of `data/preprocessing.py`.

In [ ]:
import html, re, random
from datasets import load_dataset, Dataset

_TAG = re.compile(r'<[^>]+>'); _WS = re.compile(r'[ \t]+'); _NL = re.compile(r'\n{3,}')

def clean(t):
    if not isinstance(t, str):
        t = '' if t is None else str(t)
    t = html.unescape(t); t = _TAG.sub(' ', t)
    t = _WS.sub(' ', t.replace('\r', '\n')); t = _NL.sub('\n\n', t)
    return '\n'.join(line.strip() for line in t.split('\n')).strip()

def load_pairs():
    try:
        df = load_dataset('lavita/MedQuAD', split='train').to_pandas()
        cols = {c.lower(): c for c in df.columns}
        df = df.rename(columns={cols['question']: 'question', cols['answer']: 'answer'})
        print('Loaded MedQuAD:', len(df))
    except Exception as e:
        print('MedQuAD failed (', e, ') -> PubMedQA fallback')
        df = load_dataset('pubmed_qa', 'pqa_labeled', split='train').to_pandas()
        ans = 'long_answer' if 'long_answer' in df.columns else 'final_decision'
        df = df.rename(columns={ans: 'answer'})
    return df[['question', 'answer']]

df = load_pairs()
if MAX_SAMPLES:
    df = df.head(MAX_SAMPLES)

df = df.dropna(subset=['question', 'answer'])
df['question'] = df['question'].map(clean)
df['answer'] = df['answer'].map(clean)
df = df.drop_duplicates(subset=['question'])
df = df[(df['question'].str.len() >= 12) & (df['answer'].str.len().between(20, 4000))]

examples = [
    {'instruction': INSTRUCTION, 'input': q, 'output': a}
    for q, a in zip(df['question'], df['answer'])
]
random.Random(SEED).shuffle(examples)
n_val = max(1, int(len(examples) * 0.1))
val_examples, train_examples = examples[:n_val], examples[n_val:]
print(f'train={len(train_examples)}  val={len(val_examples)}')
print('\nExample:', train_examples[0])

## 6. Load the 4-bit base model + attach LoRA adapters

This is the **QLoRA** core: NF4 4-bit frozen base + double quantization + tiny trainable LoRA adapters.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto',
    torch_dtype=compute_dtype, trust_remote_code=True, attn_implementation='eager',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Only target modules that actually exist in this architecture.
present = {n.split('.')[-1] for n, _ in model.named_modules()}
targets = [m for m in TARGET_MODULES if m in present]
print('LoRA target modules:', targets)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM', target_modules=targets,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Format data with the chat template + build the trainer

In [ ]:
from trl import SFTConfig, SFTTrainer

def render(example):
    user = f"{example['instruction']}\n\n{example['input']}".strip()
    messages = [
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': example['output']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_examples).map(render, remove_columns=['instruction', 'input', 'output'])
val_ds = Dataset.from_list(val_examples).map(render, remove_columns=['instruction', 'input', 'output'])
print(train_ds[0]['text'][:600])

use_bf16 = torch.cuda.is_bf16_supported()
args = SFTConfig(
    output_dir=OUTPUT_DIR, seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE, lr_scheduler_type='cosine', warmup_ratio=0.03,
    weight_decay=0.001, max_grad_norm=0.3, optim='paged_adamw_8bit',
    bf16=use_bf16, fp16=not use_bf16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    max_seq_length=MAX_SEQ_LENGTH, packing=False,
    logging_steps=10, eval_strategy='steps', eval_steps=50, save_steps=50,
    save_total_limit=2, report_to='none', dataset_text_field='text',
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer)
print('Trainer ready.')

## 8. Train 🚀

In [ ]:
import time
t0 = time.time()
trainer.train()
print(f'\n✅ Trained in {(time.time()-t0)/60:.1f} min')
print('Peak GPU memory: %.2f GB' % (torch.cuda.max_memory_allocated()/1e9))

## 9. Save the adapter + zip it for download

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter to', OUTPUT_DIR)
!ls -la {OUTPUT_DIR}

import shutil
shutil.make_archive('healthcare-qlora-adapter', 'zip', OUTPUT_DIR)
print('\nZipped -> healthcare-qlora-adapter.zip')
try:
    from google.colab import files
    files.download('healthcare-qlora-adapter.zip')
except Exception as e:
    print('Auto-download unavailable (', e, '). Download it from the Files panel on the left.')

## 10. Quick sanity test: base vs fine-tuned

In [ ]:
def ask(question, use_adapter=True):
    if hasattr(model, 'disable_adapter') and not use_adapter:
        ctx = model.disable_adapter()
    else:
        import contextlib; ctx = contextlib.nullcontext()
    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': f'{INSTRUCTION}\n\n{question}'}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with ctx, torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True,
                             temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

q = 'What are the early symptoms of type 2 diabetes?'
print('=== BASE ===\n', ask(q, use_adapter=False))
print('\n=== FINE-TUNED ===\n', ask(q, use_adapter=True))

## 11. Use the adapter locally

1. Unzip `healthcare-qlora-adapter.zip` into your project so you have:
   `outputs/healthcare-qlora-adapter/` (containing `adapter_config.json`, `adapter_model.safetensors`, tokenizer files).
2. In `.env`, set the base model back to the one you trained on, e.g.
   `BASE_MODEL=microsoft/Phi-3-mini-4k-instruct`.
   ⚠️ Serving the full 3.8B model on CPU is slow and may exceed the Docker VM's RAM — prefer a GPU machine, or raise Docker's memory to 12 GB+.
3. Restart serving: `docker compose up -d` (or `make api`).
4. Check it picked up the adapter: `curl localhost:8000/model-info` → `"used_adapter": true`.

To run the project's full evaluation (base vs fine-tuned) locally: `make evaluate`.